# Silver Transformation
Clean, deduplicate, and normalize bronze tables into silver Delta tables with proper schemas and relationships.

In [ ]:
# Silver: sales_orders
# Join SalesOrderHeader + SalesOrderDetail, filter null order IDs

header = spark.table("bronze_SalesOrderHeader")
detail = spark.table("bronze_SalesOrderDetail")

sales_orders = header.join(detail, header.SalesOrderID == detail.SalesOrderID, "inner") \
    .filter(header.SalesOrderID.isNotNull()) \
    .select(
        header.SalesOrderID,
        header.OrderDate,
        header.CustomerID,
        header.TotalDue,
        detail.SalesOrderDetailID,
        detail.ProductID,
        detail.OrderQty,
        detail.UnitPrice,
        detail.LineTotal
    ) \
    .dropDuplicates()

sales_orders.write.format("delta").mode("overwrite").saveAsTable("silver_sales_orders")
print(f"silver_sales_orders: {sales_orders.count()} rows")

In [ ]:
# Silver: products
# Join Product + ProductSubcategory + ProductCategory with flattened hierarchy

product = spark.table("bronze_Product")
subcategory = spark.table("bronze_ProductSubcategory")
category = spark.table("bronze_ProductCategory")

products = product \
    .join(subcategory, product.ProductSubcategoryID == subcategory.ProductSubcategoryID, "left") \
    .join(category, subcategory.ProductCategoryID == category.ProductCategoryID, "left") \
    .select(
        product.ProductID,
        product.Name.alias("ProductName"),
        product.ProductNumber,
        product.Color,
        product.ListPrice,
        subcategory.Name.alias("SubcategoryName"),
        category.Name.alias("CategoryName")
    ) \
    .dropDuplicates()

products.write.format("delta").mode("overwrite").saveAsTable("silver_products")
print(f"silver_products: {products.count()} rows")

In [ ]:
# Silver: customers_with_address
# Join Customer + CustomerAddress + Address

customer = spark.table("bronze_Customer")
customer_address = spark.table("bronze_CustomerAddress")
address = spark.table("bronze_Address")

customers = customer \
    .join(customer_address, customer.CustomerID == customer_address.CustomerID, "left") \
    .join(address, customer_address.AddressID == address.AddressID, "left") \
    .select(
        customer.CustomerID,
        customer.FirstName,
        customer.LastName,
        address.City,
        address.StateProvince,
        address.CountryRegion
    ) \
    .dropDuplicates()

customers.write.format("delta").mode("overwrite").saveAsTable("silver_customers_with_address")
print(f"silver_customers_with_address: {customers.count()} rows")

In [ ]:
# Verify silver tables
silver_tables = [t for t in spark.catalog.listTables() if t.name.startswith("silver_")]
print(f"Silver tables created: {len(silver_tables)}")
for t in silver_tables:
    count = spark.table(t.name).count()
    print(f"  {t.name}: {count} rows")